In [ ]:
import numpy as np
import bacco

import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [ ]:
def distance(x, center):
    return np.sqrt(np.sum((x - center)**2, axis=1))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

_snap = 264
zoom = {}

for i in range(len(name_list)):
    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/"+name_list[i]+"/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(_snap,_snap), dm_file="snapdir_{:03d}/snapshot_{:03d}".format(_snap,_snap),\
			    sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, use_ids=False, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(_snap,_snap), numpart=4320**3)

In [ ]:
### Load the Halo Selection ###
xmatch = {}
for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], 264, name=name_list[i])

In [ ]:
def get_contamination_fraction(zoom, xmatch):

    # The high-resolution mass will be sum of the DM, gas, stars and BH mass
    hd_mass = (zoom.fof['halo_mfof_type'][:,0] + zoom.fof['halo_mfof_type'][:,1] + zoom.fof['halo_mfof_type'][:,4] + zoom.fof['halo_mfof_type'][:,5])[xmatch['ind']]

    # The low resolution mass will be the sum of the low-res DM particles
    lr_mass = zoom.fof['halo_mfof_type'][:,2][xmatch['ind']] + zoom.fof['halo_mfof_type'][:,3][xmatch['ind']]

    return lr_mass / (hd_mass + lr_mass)    

def get_binned_contamination_fraction(zoom, xmatch, mass_bins):
    # Compute masses
    hd_mass = 1e10 * (zoom.fof['halo_mfof_type'][:,0] + zoom.fof['halo_mfof_type'][:,1] +
               zoom.fof['halo_mfof_type'][:,4] + zoom.fof['halo_mfof_type'][:,5])[xmatch['ind']]
    lr_mass = 1e10 * (zoom.fof['halo_mfof_type'][:,2][xmatch['ind']] +
               zoom.fof['halo_mfof_type'][:,3][xmatch['ind']])
    
    total_mass = hd_mass + lr_mass
    contamination = lr_mass / total_mass

    # Bin the halos by total_mass
    bin_indices = np.digitize(total_mass, mass_bins) - 1  # bins are 0-indexed
    binned_fraction = []
    bin_centers = 0.5 * (mass_bins[:-1] + mass_bins[1:])

    for i in range(len(mass_bins) - 1):
        in_bin = bin_indices == i
        if np.any(in_bin):
            binned_fraction.append(np.mean(contamination[in_bin]))
        else:
            binned_fraction.append(np.nan)  # or 0, or skip

    return bin_centers, np.array(binned_fraction)

In [ ]:
f_cont = []
m_halo = []
for i in range(len(name_list)-1):
    f_cont.append(get_contamination_fraction(zoom[name_list[i]], xmatch[name_list[i]]))
    m_halo.append(1e10 * zoom[name_list[i]].fof['halo_m200c'][xmatch[name_list[i]]['ind']])

f_cont = np.ndarray.flatten(np.array(f_cont))
m_halo = np.ndarray.flatten(np.array(m_halo))

# # High-res simulation
# f_cont_hd = get_contamination_fraction(zoom['fiducial_new_ICs'], xmatch['fiducial_new_ICs'])
# m_halo_hd = 1e10 * zoom['fiducial_new_ICs'].fof['halo_mfof'][xmatch['fiducial_new_ICs']['ind']]

In [ ]:
mass_bins = np.logspace(10, 15, 20)  # Example mass bins from 10^10 to 10^15 Msun
f_binned = {}

for i in range(len(name_list)):
    f_binned[name_list[i]] = get_binned_contamination_fraction(zoom[name_list[i]], xmatch[name_list[i]], mass_bins)

In [ ]:
print(len(f_cont[f_cont == 0]) / len(f_cont))
print(len(f_cont[f_cont <= 1e-3]) / len(f_cont))
print(len(f_cont[f_cont <= 1e-2]) / len(f_cont))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5,5), dpi=200)

ax.set_xscale('log')

ax.hist(f_cont+1e-6, bins=np.logspace(-7,0,50), density=False, log=True, histtype='step', color='C0', lw=3);
# ax.hist(f_cont_hd+1e-7, bins=np.logspace(-7,0,50), density=False, log=True, histtype='step', color='C3', lw=3);

ax.axvline(1e-3, color='k', ls='--', lw=4)
ax.axvline(1e-2, color='k', ls=':', lw=4)

ax.set_ylabel("Number of Halos", fontsize=16)
ax.set_xlabel("Contamination Fraction", fontsize=16)

ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, width=1.5, length=3, labelsize=14)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Adjust thickness as desired

# Calculate fractions
frac_1e3 = np.sum(f_cont < 1e-3) / len(f_cont) * 100
frac_1e2 = np.sum(f_cont < 1e-2) / len(f_cont) * 100

# Plot vertical lines
ax.axvline(1e-3, color='k', ls='--', lw=4)
ax.axvline(1e-2, color='k', ls=':', lw=4)

# Add labels
ax.text(3e-4, ax.get_ylim()[1]*0.1, 
        fr'$f_{{\rm cont}} < 10^{{-3}}$ ({frac_1e3:.1f}\%)', 
        rotation=90, va='center', ha='left', fontsize=11, color='k', backgroundcolor='w')

ax.text(3e-3, ax.get_ylim()[1]*0.1, 
        fr'$f_{{\rm cont}} < 10^{{-2}}$ ({frac_1e2:.1f}\%)', 
        rotation=90, va='center', ha='left', fontsize=11, color='k', backgroundcolor='w')

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/Notebooks/validation/contamination_histogram.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5,5), dpi=150)

ax.set_xscale('log')
ax.set_yscale('log')
for i in range(len(name_list)-1):
    shift = np.random.uniform(-0.1, 0.1, size=f_binned[name_list[i]][0].shape)  # Small random shift
    ax.plot(f_binned[name_list[i]][0] * (1 + shift), f_binned[name_list[i]][1]+1e-7, color='C0', lw=3, label=name_list[i], ls='', marker='o', ms=3)

ax.plot(f_binned['fiducial'][0], f_binned['fiducial'][1], color='C0', lw=3, label='Fiducial', ls='', marker='o', ms=3)
ax.fill_between(np.logspace(9,16,10), -0.5, 1e-2, color='gray', alpha=0.2, ec=None)

ax.set_xlim(1e10, 1e15)
ax.set_ylim(0,1)

ax.set_xlabel("Total Halo Mass [$M_\odot/h$]", fontsize=16)
ax.set_ylabel("Mean Contamination Fraction", fontsize=16)

ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, width=1.5, length=3, labelsize=14)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Adjust thickness as desired

In [ ]:
m_median = np.median(zoom['fiducial'].lowres_dm['mass']) * 1e10

In [ ]:
fig, ax = plt.subplots(figsize=(5.5,5), dpi=150)

ax.set_xscale('log')
ax.set_yscale('log')

m_array = np.logspace(9,16,10)
ax.fill_between(m_array, -0.5, 1e-2, color='gray', alpha=0.2, ec=None)
ax.scatter(m_halo, f_cont + 1e-6, s=1, color="k")
# ax.scatter(m_halo_hd, f_cont_hd + 1e-6, s=1, color="C3", label="High-res")
ax.plot(m_array, m_median/m_array, color='C0', lw=2, label=r"$M_\mathrm{DM,LR}/M_\mathrm{halo}$")
ax.scatter(fid_mhalo, fid_fcont+1e-6, s=2, color='C3')

ax.set_xlim(1e10, 1e15)
ax.set_ylim(1e-7,1)

ax.set_xlabel("Total Halo Mass [$M_\odot/h$]", fontsize=16)
ax.set_ylabel("Contamination Fraction", fontsize=16)

ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, width=1.5, length=3, labelsize=14)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Adjust thickness as desired

ax.legend(loc="upper right")

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/Notebooks/validation/contamination_fraction_Mh.pdf", bbox_inches='tight')

## Contamination Fraction for Fiducial Zoom

In [ ]:
fid_fcont = get_contamination_fraction(zoom['fiducial'], xmatch['fiducial'])
fid_mhalo = 1e10 * zoom['fiducial'].fof['halo_m200c'][xmatch['fiducial']['ind']]

In [ ]:
local_list = np.where((fid_fcont > 0.1) & (fid_mhalo > 10**11))[0]
cont_list = xmatch['fiducial']['ind'][(fid_fcont > 0.1) & (fid_mhalo > 10**11)]

In [ ]:
def make_histogram_zoom(index, pos):
    center = zoom['fiducial'].fof['halo_pos'][index]
    r200c = zoom['fiducial'].fof['halo_r200c'][index]
    
    dparts = distance(pos, center)
    mask = dparts < 10

    xy_range = [[-3, 3], [-3, 3]]
    hist = np.histogram2d(pos[mask][:,0]-center[0], pos[mask][:,1]-center[1], bins=400, range=xy_range)
    
    return hist

def make_histogram_mtng(index, pos):
    center = mtng.fof['halo_pos'][index]
    r200c = mtng.fof['halo_r200c'][index]
    
    dparts = distance(pos, center)
    mask = dparts < 10

    xy_range = [[-3, 3], [-3, 3]]
    hist = np.histogram2d(pos[mask][:,0]-center[0], pos[mask][:,1]-center[1], bins=400, range=xy_range)
    
    return hist

### Load lite MTNG Snapshot

In [ ]:
# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

# Load Lite snapshot of the MTNG
snap = 264
numpart = int(4320**3/64)

adr = "/cosmos_storage/simulations/TNG_Family/MTNG/"

sim_format = 'TNG500'
mtng = bacco.Simulation(verbose=False,basedir=adr, halo_file='groups_%03d/fof_subhalo_tab_%03d'%(snap,snap),\
                        dm_file='64/lite_snap_%03d_mod/diluted_snapshot_%03d'%(snap,snap),\
                        tau=tau, ns=ns, sigma8=sigma8, numpart=numpart, sim_format=sim_format,\
                        use_orphans=False, total_snapshots=265, use_ids=False)

mtng.fof['halo_pos'][:,0] = (mtng.fof['halo_pos'][:,0] - 125) % 500
mtng.sub['pos'][:,0] = (mtng.sub['pos'][:,0] - 125) % 500
mtng.dm['pos'][:,0] = (mtng.dm['pos'][:,0] - 125) % 500

In [ ]:
for i in range(len(cont_list)):
    hist_dm = make_histogram_zoom(cont_list[i], zoom['fiducial'].dm['pos'])
    hist_mtng = make_histogram_mtng(final_sel[local_list[i]], mtng.dm['pos'])

    fig, ax = plt.subplots(1, 2, dpi=200, figsize=(12, 5))

    box_size=6

    # High Resolution DM

    im_dm = ax[0].imshow(1+hist_dm[0], cmap='viridis',
                    norm=LogNorm(vmin=np.min(1+hist_mtng[0]), vmax=np.max(1+hist_mtng[0])),
                    origin='lower', extent = [-box_size/2, box_size/2, -box_size/2, box_size/2])

    # ax.scatter(0,0, marker='*', color='C0', s=200, label='Halo Center')
    circle = plt.Circle((0, 0), radius=zoom['fiducial'].fof['halo_r200c'][cont_list[i]], fill=False, color='C3')
    ax[0].add_patch(circle)

    ax[0].set_xlabel("x [Mpc/h]")
    ax[0].set_ylabel("y [Mpc/h]")
    ax[0].set_title(r"Zoom $M_h={:.2f}$".format(zoom['fiducial'].fof['halo_m200c'][cont_list[i]]))

    cbar = fig.colorbar(im_dm, ax=ax[0], fraction=0.046, pad=0.04)
    cbar.set_label(r'DM Density [$M_\odot h^{-1} / ($Mpc$/h)^3$]')

    # Low Resolution DM

    im_dm = ax[1].imshow(1+hist_mtng[0], cmap='viridis',
                    norm=LogNorm(vmin=np.min(1+hist_mtng[0]), vmax=np.max(1+hist_mtng[0])),
                    origin='lower', extent = [-box_size/2, box_size/2, -box_size/2, box_size/2])

    # ax.scatter(0,0, marker='*', color='C0', s=200, label='Halo Center')
    circle = plt.Circle((0, 0), radius=mtng.fof['halo_r200c'][final_sel[local_list[i]]], fill=False, color='C3')
    ax[1].add_patch(circle)

    ax[1].set_xlabel("x [Mpc/h]")
    ax[1].set_ylabel("y [Mpc/h]")
    ax[1].set_title(r"MTNG $M_h={:.2f}$".format(mtng.fof['halo_m200c'][final_sel[local_list[i]]]))

    cbar = fig.colorbar(im_dm, ax=ax[1], fraction=0.046, pad=0.04)
    cbar.set_label(r'DM Density [$M_\odot h^{-1} / ($Mpc$/h)^3$]')

    plt.tight_layout()
    plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/Notebooks/validation/halo_plots/zoom_vs_MTNG_{:d}.pdf".format(cont_list[i]), bbox_inches='tight')

In [ ]:
print(zoom['fiducial'].fof['halo_pos'][cont_list[0]])
print(mtng.fof['halo_pos'][final_sel[local_list[0]]])

print(zoom['fiducial'].fof['halo_m200c'][cont_list[0]])
print(mtng.fof['halo_m200c'][final_sel[local_list[0]]])

In [ ]:
hist_mtng = make_histogram_mtng(final_sel[local_list[-2]], mtng.dm['pos'])

In [ ]:
fig, ax = plt.subplots(dpi=200)

box_size=20

ax.imshow(1+hist_mtng[0], cmap='viridis',
                    norm=LogNorm(vmin=np.min(1+hist_mtng[0]), vmax=np.max(1+hist_mtng[0])),
                    origin='lower', extent = [-box_size/2, box_size/2, -box_size/2, box_size/2])

# ax.scatter(0,0, marker='*', color='C0', s=200, label='Halo Center')
circle = plt.Circle((0, 0), radius=mtng.fof['halo_r200c'][final_sel[local_list[-2]]], fill=False, color='C3')
ax.add_patch(circle)
